# 03: API Requests

This notebook covers HTTP APIs, JSON data, and making API requests with Python.

**Topics covered:**
- HTTP methods (GET, POST)
- JSON parsing with the requests library
- Working with public APIs
- Error handling and retry logic
- Rate limiting concepts

In [ ]:
import requests
import json
import time

## 1. HTTP Basics

HTTP (HyperText Transfer Protocol) is the foundation of web communication. APIs use HTTP methods to perform operations.

| Method | Purpose | Example |
|--------|---------|---------|
| GET | Retrieve data | Fetch a list of users |
| POST | Send/create data | Submit a form, create a resource |
| PUT | Update data | Update a user's profile |
| DELETE | Remove data | Delete a record |

### HTTP Response Codes

| Code | Meaning |
|------|---------|
| 200 | OK — Success |
| 201 | Created — Resource created |
| 400 | Bad Request — Invalid input |
| 401 | Unauthorized — Auth required |
| 404 | Not Found — Resource doesn't exist |
| 429 | Too Many Requests — Rate limited |
| 500 | Server Error — Something broke |

## 2. Making GET Requests

We'll use JSONPlaceholder (https://jsonplaceholder.typicode.com) — a free fake API for testing.

In [ ]:
# Simple GET request
response = requests.get("https://jsonplaceholder.typicode.com/posts/1")

print(f"Status code: {response.status_code}")
print(f"Headers (content-type): {response.headers.get('content-type')}")
print(f"Response body:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# GET a list of resources
response = requests.get("https://jsonplaceholder.typicode.com/users")
users = response.json()

print(f"Number of users: {len(users)}")
for user in users[:3]:
    print(f"  - {user['name']} ({user['email']})")

In [ ]:
# Query parameters
# Get posts by a specific user
response = requests.get(
    "https://jsonplaceholder.typicode.com/posts",
    params={"userId": 1}
)
posts = response.json()
print(f"Posts by user 1: {len(posts)}")
for post in posts[:3]:
    print(f"  - {post['title'][:50]}...")

## 3. Working with JSON

JSON (JavaScript Object Notation) is the standard data format for APIs.

In [ ]:
# Parsing JSON
response = requests.get("https://jsonplaceholder.typicode.com/posts/1")
post = response.json()

# Access nested data
print(f"Post ID: {post['id']}")
print(f"Title: {post['title']}")
print(f"Body preview: {post['body'][:100]}...")

In [ ]:
# Create JSON to send
new_post = {
    "title": "AI Engineering Course",
    "body": "Learning about APIs and JSON in Python.",
    "userId": 1
}

# Convert to JSON string
json_string = json.dumps(new_post, indent=2)
print("JSON string:")
print(json_string)

# Parse back to dict
parsed = json.loads(json_string)
print(f"\nParsed title: {parsed['title']}")

## 4. Making POST Requests

POST requests send data to the server to create a new resource.

In [ ]:
# POST with JSON body
new_post = {
    "title": "Learning API Requests",
    "body": "This is a post created with a POST request.",
    "userId": 1
}

response = requests.post(
    "https://jsonplaceholder.typicode.com/posts",
    json=new_post,  # Automatically sets Content-Type: application/json
    headers={"Content-Type": "application/json"}
)

print(f"Status code: {response.status_code}")
print(f"Response:")
print(json.dumps(response.json(), indent=2))

## 5. Error Handling

Always handle errors gracefully when making API calls.

In [ ]:
# Check response status
response = requests.get("https://jsonplaceholder.typicode.com/posts/99999")

print(f"Status code: {response.status_code}")
print(f"OK: {response.ok}")

if response.ok:
    data = response.json()
else:
    print(f"Request failed with status {response.status_code}")

In [ ]:
# Comprehensive error handling
def safe_api_call(url, method="GET", **kwargs):
    """Make an API call with comprehensive error handling."""
    try:
        if method == "GET":
            response = requests.get(url, timeout=10, **kwargs)
        elif method == "POST":
            response = requests.post(url, timeout=10, **kwargs)
        else:
            raise ValueError(f"Unsupported method: {method}")
        
        response.raise_for_status()  # Raises HTTPError for 4xx/5xx
        return {"success": True, "data": response.json(), "status": response.status_code}
        
    except requests.exceptions.Timeout:
        return {"success": False, "error": "Request timed out"}
    except requests.exceptions.ConnectionError:
        return {"success": False, "error": "Connection failed"}
    except requests.exceptions.HTTPError as e:
        return {"success": False, "error": f"HTTP {e.response.status_code}: {e}"}
    except requests.exceptions.RequestException as e:
        return {"success": False, "error": str(e)}
    except json.JSONDecodeError:
        return {"success": False, "error": "Invalid JSON response"}

# Test it
result = safe_api_call("https://jsonplaceholder.typicode.com/posts/1")
print(f"Success: {result['success']}")
if result["success"]:
    print(f"Title: {result['data']['title']}")

# Test with invalid URL
result = safe_api_call("https://invalid.example.com/api")
print(f"\nFailed call: {result}")

## 6. Retry Logic with Exponential Backoff

When APIs fail transiently, retry with increasing delays.

In [ ]:
def api_call_with_retry(url, max_retries=3, base_delay=1.0):
    """Make an API call with exponential backoff retry."""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=10)
            
            # Handle rate limiting specifically
            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", base_delay * (2 ** attempt)))
                print(f"  Rate limited. Waiting {retry_after}s...")
                time.sleep(retry_after)
                continue
            
            response.raise_for_status()
            return response.json()
            
        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                raise
            delay = base_delay * (2 ** attempt)  # 1s, 2s, 4s
            print(f"  Attempt {attempt + 1} failed: {e}. Retrying in {delay}s...")
            time.sleep(delay)

# Test
try:
    data = api_call_with_retry("https://jsonplaceholder.typicode.com/posts/1")
    print(f"Success: {data['title']}")
except Exception as e:
    print(f"All retries failed: {e}")

## 7. Rate Limiting Concepts

Most APIs limit how many requests you can make in a given time period.

### Best Practices:
1. **Check rate limit headers** in responses (e.g., `X-RateLimit-Remaining`)
2. **Implement delays** between requests when needed
3. **Cache responses** to avoid redundant calls
4. **Use batch endpoints** instead of many individual requests
5. **Respect the limit** — don't try to circumvent it

In [ ]:
# Simple rate limiter
class RateLimiter:
    def __init__(self, requests_per_second=1):
        self.min_interval = 1.0 / requests_per_second
        self.last_request_time = 0
    
    def wait_if_needed(self):
        """Wait if necessary to respect rate limit."""
        elapsed = time.time() - self.last_request_time
        if elapsed < self.min_interval:
            sleep_time = self.min_interval - elapsed
            time.sleep(sleep_time)
        self.last_request_time = time.time()

# Usage
limiter = RateLimiter(requests_per_second=2)

urls = [
    "https://jsonplaceholder.typicode.com/posts/1",
    "https://jsonplaceholder.typicode.com/posts/2",
    "https://jsonplaceholder.typicode.com/posts/3",
]

results = []
for url in urls:
    limiter.wait_if_needed()
    response = requests.get(url)
    results.append(response.json()["title"])
    print(f"Fetched: {results[-1][:50]}")

print(f"\nFetched {len(results)} posts successfully")

## 8. Practical Example: Fetching and Processing Data

Let's combine API calls with data processing.

In [ ]:
import pandas as pd

# Fetch all posts
response = requests.get("https://jsonplaceholder.typicode.com/posts")
posts = response.json()

# Fetch all users
response = requests.get("https://jsonplaceholder.typicode.com/users")
users = response.json()

# Create a user lookup dictionary
user_lookup = {u["id"]: u["name"] for u in users}

# Build a DataFrame from posts
posts_df = pd.DataFrame(posts)
posts_df["author"] = posts_df["userId"].map(user_lookup)
posts_df["title_length"] = posts_df["title"].str.len()
posts_df["body_length"] = posts_df["body"].str.len()

print(f"Total posts: {len(posts_df)}")
print(posts_df[["title", "author", "title_length"]].head())

In [ ]:
# Analyze the data
author_stats = posts_df.groupby("author").agg({
    "title_length": "mean",
    "body_length": "mean",
    "id": "count"
}).rename(columns={"id": "post_count"}).round(1)

print("Posts per author:")
print(author_stats.sort_values("post_count", ascending=False))

### Exercise: API Requests

Using JSONPlaceholder API:
1. Fetch all comments and create a DataFrame with columns: postId, name, email, body
2. Find which post has the most comments
3. Create a summary showing the number of comments per post

Hint: `requests.get("https://jsonplaceholder.typicode.com/comments")`

In [ ]:
# Your code here:


## Summary

Key takeaways:

1. **HTTP methods**: GET retrieves data, POST creates data
2. **Status codes**: Always check — 200 means success, 4xx/5xx means problems
3. **JSON**: Use `response.json()` to parse, `json.dumps/loads` to convert
4. **Error handling**: Wrap API calls in try/except blocks
5. **Rate limiting**: Respect API limits with delays and backoff
6. **Combine**: API data + Pandas = powerful analysis

**Next**: [04-git-basics.ipynb](04-git-basics.ipynb)